# Phase recognition walkthrough

This notebook calls the same library functions `python -m src.phase.train` calls;
it does not re-implement training, so anything you see here is what the trainer
does.

Everything runs on `src/data/phantom.py`, a procedurally generated endoscopic phantom. It is not surgical data! I didn't have access to this. 
The step id is encoded in the mucosa tint and in which instrument pair is on screen, with a
per-video jitter, so the task is learnable but only by generalising across
videos. Read every score below as "the pipeline works", not as performance for clinical, which you must work on.

## 0. Setup

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
os.environ["PSAI_FORCE_CPU"] = "1"

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.common.device import device_report, get_device
from src.common.seed import seed_everything

plt.rcParams["figure.dpi"] = 58
seed_everything(0)
device = get_device()
print(device_report(device))

ROOT = Path(tempfile.mkdtemp(prefix="psai-phase-nb-"))
print("scratch data root:", ROOT)

## 1. The phantom

In [ ]:
from src.data.phantom import N_INSTRUMENT_KINDS, N_STEPS, render_labelled_frame

frames = [render_labelled_frame(np.random.default_rng(s), size=128) for s in range(4)]

fig, axes = plt.subplots(2, 4, figsize=(11, 5.4))
for i, f in enumerate(frames):
    axes[0, i].imshow(f.image)
    axes[0, i].set_title(f"step {f.step} | tools {list(f.instruments)}", fontsize=9)
    axes[1, i].imshow(f.mask, cmap="gray")
    axes[1, i].set_title(f"mask ({(f.mask > 0).mean():.1%} of frame)", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"{N_STEPS} steps, {N_INSTRUMENT_KINDS} instrument kinds")
print("presence vector of frame 0:", frames[0].presence(N_INSTRUMENT_KINDS))

## 2. On-disk videos + the split

In [ ]:
from src.data.make_dummy_data import make_phase
from src.data.splits import load_splits

make_phase(ROOT / "phase", n_videos=12, n_frames=48, size=64, seed=0)

splits = load_splits(ROOT / "phase")
print()
for name, ids in splits.items():
    print(f"{name:5s} {ids}")
print("train n val:", set(splits["train"]) & set(splits["val"]))
print("train n test:", set(splits["train"]) & set(splits["test"]))

## 3. `ClipDataset`

Every sample carries `valid` (per-frame bool) and `frame_index` (absolute, -1
where padded). Windows cover the whole video including the tail.

In [ ]:
from src.phase.dataset import ClipDataset, window_starts

ds = ClipDataset(str(ROOT / "phase"), clip_len=24, stride=12, img_size=64,
                 split=splits["train"], split_name="train")
item = ds[0]
for k, v in item.items():
    print(f"  {k:12s} {tuple(v.shape) if torch.is_tensor(v) else v}")

print("\nwindow starts for a 48-frame video, clip_len=24, stride=12:",
      window_starts(48, 24, 12))
covered = {f for s in window_starts(48, 24, 12) for f in range(s, s + 24)}
print("frames covered:", len(covered & set(range(48))), "of 48")
print("padded frames in this sample:", int((~item["valid"]).sum()))

In [ ]:
# Label timeline of one clip
clip = ds[0]
steps = clip["step"].numpy()
instr = clip["instrument"].numpy().T

fig, (a0, a1) = plt.subplots(2, 1, figsize=(10, 4.2),
                             gridspec_kw={"height_ratios": [1, 3]})
a0.imshow(steps[None, :], aspect="auto", cmap="tab20", vmin=0, vmax=13)
a0.set_yticks([]); a0.set_title("step label per frame")
a1.imshow(instr, aspect="auto", cmap="Greys", interpolation="nearest")
a1.set_ylabel("instrument kind"); a1.set_xlabel("frame")
a1.set_title("instrument presence")
plt.tight_layout()
plt.show()

## 4. Model

In [ ]:
from src.common.lora import count_trainable
from src.phase.model import PhaseModelConfig, SpatioTemporalMultiTask
from src.phase.train import PhaseTrainConfig, class_weights

cfg = PhaseTrainConfig(
    data_root=str(ROOT / "phase"), splits="splits.json",
    backbone="timm:resnet18", pretrained=False, temporal="tcn",
    temporal_ch=128, temporal_layers=4, clip_len=24, stride=12, img_size=64,
    lr=3.0e-4, epochs=12, batch_size=2, workers=0, seed=0)

step_counts, instr_pos, n_frames = ds.label_counts()
step_w, pos_w = class_weights(step_counts, instr_pos, n_frames)
print(f"class weights from {n_frames} TRAIN frames only "
      f"(step {min(step_w):.2f}..{max(step_w):.2f})")

model_cfg = PhaseModelConfig.from_config(cfg, step_class_weight=step_w,
                                         instrument_pos_weight=pos_w)
model = SpatioTemporalMultiTask(model_cfg).to(device)

trainable, total = count_trainable(model)
print(f"\nbackend {model.backend} | feature dim {model.encoder.feat_dim} | "
      f"{trainable:,} / {total:,} params trainable")

## 5. Padding cannot reach a real frame

In [ ]:
short = torch.stack([ds[0]["clip"]])
valid = torch.stack([ds[0]["valid"]]).clone()
valid[:, 12:] = False  # pretend the clip ended at frame 12

model.eval()
with torch.no_grad():
    a = model(short.to(device), valid.to(device))["step"][0, :12]
    noisy = short.clone()
    noisy[:, 12:] = torch.randn_like(noisy[:, 12:])
    b = model(noisy.to(device), valid.to(device))["step"][0, :12]

print("max change in real-frame logits after scrambling every padded frame:",
      float((a - b).abs().max()))

## 6. Train

`train_one_epoch` / `evaluate` are imported from the trainer.

In [ ]:
from src.phase.train import (build_dataset, build_loader, build_scheduler, evaluate,
                             save_checkpoint, train_one_epoch)

train_ds = build_dataset(cfg, splits["train"], "train")
val_ds = build_dataset(cfg, splits["val"], "val")
train_loader = build_loader(train_ds, cfg, shuffle=True)
val_loader = build_loader(val_ds, cfg, shuffle=False)
print(f"train {len(splits['train'])} videos / {len(train_ds)} clips | "
      f"val {len(splits['val'])} videos / {len(val_ds)} clips {splits['val']}")

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = build_scheduler(opt, cfg, len(train_loader))

CKPT = str(ROOT / "best.pt")
history, best = [], -1.0
for epoch in range(cfg.epochs):
    loss = train_one_epoch(model, train_loader, opt, sched, device, cfg, epoch)
    score = evaluate(model, val_loader, device, cfg, "val")
    history.append((loss, score))
    print(f"epoch {epoch:02d} | {model.backend} | train loss {loss:.4f} | {score.describe()}")
    if score.step_macro_f1 > best:  # val selects the checkpoint, exactly as train.py does
        best = score.step_macro_f1
        save_checkpoint(CKPT, model, model_cfg, cfg, splits, epoch, score)

In [ ]:
losses = [h[0] for h in history]
f1s = [h[1].step_macro_f1 for h in history]
accs = [h[1].step_acc for h in history]

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.6))
a0.plot(losses); a0.set_title("train loss"); a0.set_xlabel("epoch"); a0.grid(alpha=0.3)
a1.plot(f1s, label="val step macro-F1")
a1.plot(accs, label="val step accuracy")
a1.axhline(1 / cfg.num_steps, ls="--", c="gray", label="chance accuracy (1/14)")
a1.set_xlabel("epoch"); a1.legend(); a1.grid(alpha=0.3)
a1.set_title("held-out validation videos")
plt.tight_layout()
plt.show()

## 7. Segmental metrics

NOTE: Frame-wise F1 can't tell a clean prediction from one that flickers between
steps every few frames, so having a MS-TCN edit score and a F1@k separates them.

In [ ]:
from src.phase.metrics import macro_f1_multiclass, segmental_edit_score, segmental_f1_at_k

truth = np.repeat([0, 1, 2, 3], 25)
clean = np.repeat([0, 1, 2, 3], 25).copy()
clean[:6] = 1
flicker = truth.copy()
flicker[::7] = (flicker[::7] + 1) % 4

for name, pred in (("clean", clean), ("flickering", flicker)):
    print(f"{name:11s} frame macro-F1 {macro_f1_multiclass(pred, truth, 4, from_logits=False):.3f}"
          f" | edit {segmental_edit_score(pred, truth):5.1f}"
          f" | F1@10 {segmental_f1_at_k(pred, truth, 0.10):5.1f}")

## 8. Checkpoint round trip

In [ ]:
from src.phase.train import load_checkpoint

reloaded, reloaded_cfg, payload = load_checkpoint(CKPT)
rescored = evaluate(reloaded.to(device), val_loader, device, cfg, "val")

print("epoch saved:", payload["epoch"], "| selected by:", payload["selected_by"])
print("saved   val step macro-F1:", round(payload["score"]["step_macro_f1"], 6))
print("reload  val step macro-F1:", round(rescored.step_macro_f1, 6))
print("split recorded in the checkpoint:", payload["splits"]["val"])
print("instrument_loss_weight recorded in the checkpoint:",
      payload["model_cfg"]["instrument_loss_weight"])

## 9. Held-out test

In [ ]:
test_loader = build_loader(build_dataset(cfg, splits["test"], "test"), cfg, shuffle=False)
test_score = evaluate(reloaded.to(device), test_loader, device, cfg, "test")
print(f"{reloaded.backend} | {test_score.describe()} — held out from both training "
      "and checkpoint selection")